# 4 - CHUNK IDENTIFICATION: can bge find the PASSAGE, not just the page?

Notebook 3 settled the document question: chunk-max gets the right page into the top 20 for
**98.4%** of questions. A reader cannot use a page - it needs the paragraph. This notebook
scores all **469,387 chunks** globally and asks where the chunk that actually contains the
answer lands.

No re-embedding: `chunk_emb.npy` is pulled from HF. Only the 16,229 queries are encoded.

**What comes out of this, and why each piece matters:**

| output | used for |
|---|---|
| recall@1..100 curve | picking the rerank depth from data instead of convention |
| **recall@100** | the reranker's hard ceiling - it can only reorder what it is given |
| context budget (k x 200 words) | whether the reader gets 3 chunks or 20 |
| page-found-but-chunk-missed split | whether a reranker can help at all, or retrieval is the problem |
| `top100_chunks.npy` | the reranker's candidate pool AND its hard negatives |

Still no val split: nothing here is tuned, every configuration is reported. The split
comes due at the reranker, which IS a trained model.

In [ ]:
# §0 SETUP
import os, sys, json, pickle, time
sys.path.insert(0, os.getcwd())
import numpy as np
import torch
import nqrun
from sentence_transformers import SentenceTransformer

In [ ]:
# §1 CONFIG - must match notebook 3 exactly, or the saved chunk matrix does not describe these
# chunks. §2 asserts the match rather than trusting it.
MODEL   = 'BAAI/bge-base-en-v1.5'
CHUNK   = 200
OVERLAP = 50
STRIDE  = CHUNK - OVERLAP        # 150
KS      = (1, 3, 5, 10, 20, 50, 100)
TOPK    = 100                    # candidate pool saved for the reranker
BATCH   = 256
MAXT    = 5                      # max answer length in words (notebook 1)

QPREFIX = 'Represent this sentence for searching relevant passages: '   # measured in notebook 3

# Notebook 3's document-level results, for the comparison table at the end.
DOC_MAX_DOC = {1: 0.643, 5: 0.911, 10: 0.961, 20: 0.984}

N_PAGES_EXPECTED, N_QS_EXPECTED, N_CHUNKS_EXPECTED = 18997, 16229, 469387
OUT, HF_DIR = nqrun.art('step2-retrieval'), 'step2-retrieval'
print(f'{MODEL} | chunk {CHUNK}/{OVERLAP} stride {STRIDE} | top-{TOPK}')
print(f'root {nqrun.ROOT} | run {nqrun.RUN_ID} | {nqrun.gpu_info()}')

## Definitions

Nothing below executes; running these cells only defines names.

In [ ]:
# DEFINITIONS — reads the module-level `base` and `n_per_page`, which the
# orchestration below derives from the saved chunk->page map.
def chunks_for_span(p, lo, hi):
    """Global chunk ids on page p that wholly contain [lo, hi)."""
    j0 = max(0, -(-(hi - CHUNK) // STRIDE))        # smallest j with hi <= j*S + C
    j1 = min(lo // STRIDE, n_per_page[p] - 1)      # largest  j with j*S <= lo
    return [int(base[p] + j) for j in range(j0, j1 + 1)]

## Orchestration

In [ ]:
# §2 LOAD + GUARD. Corpus and notebook 3's chunk matrix, both from HF (the local
# step1-clean pickle is a shard-0 file).
C = pickle.load(open(nqrun.hf_pull('step1-clean/clean_corpus.pkl'), 'rb'))
docs, questions = C['documents'], C['questions']
E = np.load(nqrun.hf_pull(f'{HF_DIR}/chunk_emb.npy'), mmap_mode='r')
IDX = json.load(open(nqrun.hf_pull(f'{HF_DIR}/chunk_index.json')))
H, owner = IDX['doc_hashes'], np.array(IDX['owner'], dtype='int32')

assert len(docs) == N_PAGES_EXPECTED, f'{len(docs)} pages - WRONG ARTIFACT'
assert len(questions) == N_QS_EXPECTED, f'{len(questions)} questions - WRONG ARTIFACT'
assert E.shape[0] == N_CHUNKS_EXPECTED == len(owner), f'{E.shape} vs {len(owner)} owners'
assert IDX['model'] == MODEL and IDX['chunk'] == CHUNK and IDX['overlap'] == OVERLAP, \
    f"chunk matrix was built with {IDX['model']} {IDX['chunk']}/{IDX['overlap']}"
assert H == list(docs), 'page order differs from the matrix - rows would be misattributed'
print(f'{len(docs):,} pages | {len(questions):,} questions | {E.shape[0]:,} chunks | guards OK')

In [ ]:
# §3 GOLD CHUNKS - derived, not guessed.
#
# Notebook 3 appended chunks page by page in H order, starts 0, STRIDE, 2*STRIDE, ... while the
# slice was non-empty. So page p's chunk j is global index base[p] + j, and it covers
# words [j*STRIDE, j*STRIDE + CHUNK). The starts were never stored, but the stride fully
# determines them - and the assert below proves the reconstruction against the real counts.
n_per_page = np.bincount(owner, minlength=len(H))
base = np.concatenate([[0], np.cumsum(n_per_page)[:-1]]).astype('int64')
expected = np.array([max(1, -(-len(docs[h]['words']) // STRIDE)) for h in H])
assert (n_per_page == expected).all(), 'chunk counts do not match ceil(words/stride)'

# A span [lo, hi) sits wholly inside chunk j iff  j*STRIDE <= lo  and  hi <= j*STRIDE+CHUNK.
# At least one such j always exists while OVERLAP > MAXT: take j = lo//STRIDE, then
# lo - j*STRIDE <= STRIDE-1, so hi - j*STRIDE <= (STRIDE-1) + MAXT < CHUNK.
assert OVERLAP > MAXT, 'overlap must exceed the max answer length or a span can straddle'

row = {h: i for i, h in enumerate(H)}
# gold_sets pools chunks across ALL of a question's spans: a question is answerable from
# ANY chunk holding ANY of its valid answers, so retrieving any one of them is a success.
# NQ gives up to 8 spans for list-type questions ('who won the super bowl 2 years in a
# row' has 7, one per team), which is why the pooled set can be large.
gold_sets, per_span = [], []
for q in questions:
    p = row[q['doc_hash']]
    spans = [chunks_for_span(p, lo, hi) for _, lo, hi in q['gold']]
    per_span.append(spans)
    gold_sets.append({c for s in spans for c in s})

n_gold = np.array([len(s) for s in gold_sets])
print(f'questions with NO gold chunk : {(n_gold == 0).sum()}        <- must be 0')
print(f'gold chunks per question     : min {n_gold.min()} | median {int(np.median(n_gold))} '
      f'| max {n_gold.max()} | mean {n_gold.mean():.2f}')
print(f'spans per question           : mean {np.mean([len(s) for s in per_span]):.2f} '
      f'| max {max(len(s) for s in per_span)}')
assert (n_gold > 0).all(), 'a span straddles every boundary - impossible by the proof above'

# Spot-check PER SPAN: chunk c derived from span k must contain span k's text. Checking
# span 0's text against chunks derived from span 5 is meaningless - a multi-answer question
# spreads its spans across the page, so those chunks need not contain the first answer.
bad = []
for i in np.random.default_rng(0).choice(len(questions), 500, replace=False):
    q = questions[i]
    w = docs[q['doc_hash']]['words']
    p = row[q['doc_hash']]
    for k, (_, lo, hi) in enumerate(q['gold']):
        target = q['answers_rendered'][k]
        for c in chunks_for_span(p, lo, hi):          # independent re-derivation
            j = c - base[p]
            if target not in ' '.join(w[j * STRIDE:j * STRIDE + CHUNK]):
                bad.append((int(i), int(c), k, target))
print(f'spot-check (500 sampled, per span): mismatches {len(bad)}  <- must be 0')
assert not bad, f'gold chunk derivation is wrong, e.g. {bad[:3]}'

In [ ]:
# §4 SCORE every question against all 469,387 chunks. Keeps the top-TOPK ids (the
# reranker's pool) AND the exact rank of the best gold chunk, which needs no cutoff.

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
m = SentenceTransformer(MODEL, device=dev)
m.max_seq_length = 512
qtext = [q['question'] for q in questions]
Q = np.asarray(m.encode([QPREFIX + t for t in qtext], batch_size=BATCH,
                        normalize_embeddings=True, show_progress_bar=True), dtype='float32')

Et = torch.from_numpy(np.ascontiguousarray(E)).to(dev)
own_t = torch.from_numpy(owner.astype('int64')).to(dev)
top100 = np.empty((len(Q), TOPK), dtype='int32')
crank = np.empty(len(Q), dtype='int32')      # rank of the best GOLD CHUNK
prank = np.empty(len(Q), dtype='int32')      # rank of the gold PAGE under chunk-max
gi_page = np.array([row[q['doc_hash']] for q in questions])

t0, BLK = time.time(), 128
for a in range(0, len(Q), BLK):
    q = torch.from_numpy(Q[a:a + BLK]).to(dev)
    S = q @ Et.T                                                   # (b, n_chunks)
    top100[a:a + BLK] = torch.topk(S, TOPK, dim=1).indices.cpu().numpy()
    # chunk rank: how many chunks beat this question's BEST gold chunk
    for r in range(S.shape[0]):
        g = S[r, list(gold_sets[a + r])].max()
        crank[a + r] = int((S[r] > g).sum().item())
    # page rank under chunk-max, from the same scores - free decomposition
    page = torch.full((S.shape[0], len(H)), -2.0, device=dev)
    page.scatter_reduce_(1, own_t.expand(S.shape[0], -1), S, reduce='amax')
    gp = page.gather(1, torch.from_numpy(gi_page[a:a + BLK].astype('int64')).to(dev)[:, None])
    prank[a:a + BLK] = (page > gp).sum(1).cpu().numpy()
    del S, page
    if a % (BLK * 20) == 0:
        print(f'  {a:,}/{len(Q):,}  {time.time() - t0:.0f}s', flush=True)
print(f'scored in {time.time() - t0:.0f}s')

In [ ]:
# §5 THE CURVE. chunk recall@k = the gold chunk is within the top k chunks.
print(f"{'k':>5}{'chunk recall':>15}{'doc recall (nb 3)':>18}{'context words':>15}")
res = {'chunk_recall': {}, 'doc_recall_recomputed': {}}
for k in KS:
    cr = float((crank < k).mean())
    res['chunk_recall'][k] = cr
    dr = f"{DOC_MAX_DOC[k]:.1%}" if k in DOC_MAX_DOC else '-'
    print(f'{k:>5}{cr:>15.1%}{dr:>18}{k * CHUNK:>15,}')
res['chunk_mrr'] = float((1.0 / (crank + 1)).mean())
print(f"\nchunk MRR {res['chunk_mrr']:.3f} | median chunk rank {int(np.median(crank))}")

# sanity: the page rank recomputed here must reproduce notebook 3's chunk-max row
for k in (1, 5, 10, 20):
    res['doc_recall_recomputed'][k] = float((prank < k).mean())
print('\ndoc recall recomputed from the same scores (must match notebook 3 chunk-max):')
print('  ' + '  '.join(f'R@{k} {res["doc_recall_recomputed"][k]:.1%}' for k in (1, 5, 10, 20)))

print('\n--- WHERE THE LOSS IS, at k=100 ---')
page_ok = prank < 20
chunk_ok = crank < TOPK
print(f'  page in top-20 AND chunk in top-100 : {(page_ok & chunk_ok).mean():>6.1%}')
print(f'  page in top-20 but chunk NOT        : {(page_ok & ~chunk_ok).mean():>6.1%}'
      f'   <- a reranker cannot fix this either; the chunk was never retrieved')
print(f'  page NOT in top-20                  : {(~page_ok).mean():>6.1%}'
      f'   <- document-level failure')
res['page_ok_chunk_ok'] = float((page_ok & chunk_ok).mean())
res['page_ok_chunk_missed'] = float((page_ok & ~chunk_ok).mean())
res['page_missed'] = float((~page_ok).mean())

print('\n--- RERANK DEPTH: what each extra candidate buys ---')
prev = 0.0
for k in KS:
    cr = res['chunk_recall'][k]
    print(f'  depth {k:>4}: ceiling {cr:>6.1%}  (+{cr - prev:>5.1%} over depth below)  '
          f'{k * len(questions) / 1e6:.2f}M cross-encoder passes')
    prev = cr

In [ ]:
# §6 FAILURES. Split by cause, because the two causes need different fixes.
miss = np.where(crank >= TOPK)[0]
print(f'{len(miss):,} questions whose gold chunk is outside the top {TOPK}\n')
rng = np.random.default_rng(0)

for label, sel in [('PAGE found, chunk missed', miss[prank[miss] < 20]),
                   ('PAGE missed too', miss[prank[miss] >= 20])]:
    print(f'--- {label}: {len(sel):,} questions')
    for i in rng.choice(sel, size=min(6, len(sel)), replace=False) if len(sel) else []:
        q = questions[i]; d = docs[q['doc_hash']]
        print(f"  chunk rank {crank[i]:>7,} | page rank {prank[i]:>5,} | {q['question'][:58]}")
        print(f"      {d['title'][:46]} | answer {q['answer']!r} at word {q['gold'][0][1]:,}"
              f" of {len(d['words']):,}")
    print()

# does answer position inside the page predict failure?
pos = np.array([q['gold'][0][1] for q in questions])
plen = np.array([len(docs[q['doc_hash']]['words']) for q in questions])
print(f"{'subset':<26}{'n':>8}{'chunk R@100':>14}{'chunk R@20':>13}")
for lbl, sel in [('answer in first 200 w', pos < 200),
                 ('answer at 200-1000 w', (pos >= 200) & (pos < 1000)),
                 ('answer beyond 1000 w', pos >= 1000),
                 ('short page (<1k w)', plen < 1000),
                 ('long page (>=4k w)', plen >= 4000)]:
    print(f'{lbl:<26}{sel.sum():>8,}{(crank[sel] < 100).mean():>14.1%}'
          f'{(crank[sel] < 20).mean():>13.1%}')

In [ ]:
# §7 SAVE. top100_chunks.npy is the reranker's candidate pool and, minus the gold rows,
# its hard negatives - saving it means step 3 never repeats this scoring pass.
np.save(OUT / 'top100_chunks.npy', top100)
np.save(OUT / 'gold_chunk_rank.npy', crank)
json.dump({'gold_chunks': [sorted(s) for s in gold_sets],
           'model': MODEL, 'chunk': CHUNK, 'overlap': OVERLAP, 'topk': TOPK},
          open(OUT / 'gold_chunks.json', 'w'))
print(f"-> {OUT / 'top100_chunks.npy'} "
      f"({os.path.getsize(OUT / 'top100_chunks.npy') / 1e6:.1f} MB)")

for f, msg in [('top100_chunks.npy', f'top-{TOPK} chunk candidates per question'),
               ('gold_chunk_rank.npy', 'rank of the best gold chunk per question'),
               ('gold_chunks.json', 'gold chunk ids per question')]:
    nqrun.hf_push(str(OUT / f), f'{HF_DIR}/{f}', msg)

nqrun.log_metrics('4_chunk_recall', model=MODEL, chunk=CHUNK, overlap=OVERLAP,
                  n_chunks=int(E.shape[0]), n_questions=len(questions), topk=TOPK,
                  chunk_mrr=res['chunk_mrr'],
                  median_chunk_rank=int(np.median(crank)),
                  page_ok_chunk_ok=res['page_ok_chunk_ok'],
                  page_ok_chunk_missed=res['page_ok_chunk_missed'],
                  page_missed=res['page_missed'],
                  **{f'chunk_r@{k}': v for k, v in res['chunk_recall'].items()},
                  **{f'doc_r@{k}_recomputed': v
                     for k, v in res['doc_recall_recomputed'].items()})